In [4]:
import json
import pandas as pd
import glob
import numpy as np

# Crowds (H)

In [5]:
with open('./../2024-07-21_resolution_set.json') as file:
    rs = json.load(file) #Resolution set
resolutions = pd.json_normalize(rs, record_path=['resolutions'])
resolutions = resolutions[~resolutions['id'].apply(lambda x: isinstance(x, list))] #removed keys that are lists
resolutions = resolutions[resolutions['resolved']] #Filtered resolutions to resolved: true
print(len(resolutions))

1532


In [6]:
data_sources = ['acled', 'wikipedia', 'yfinance', 'fred', 'dbnomics']
market_sources = ['polymarket', 'manifold', 'metaculus', 'infer']
data_res = resolutions[resolutions['source'].isin(data_sources)]
market_res = resolutions[resolutions['source'].isin(market_sources)]

print("Data count:", len(data_res))
print("Market count:", len(market_res))

Data count: 1414
Market count: 118


## Superforecasters

### Question Source: Market

In [8]:
with open('../../ForecastBench-HumanGroups/2024-07-21.ForecastBench.human_super.json') as file:
    spf = json.load(file) #Superforecasters
superfs = pd.json_normalize(spf, record_path=['forecasts'])
print(len(superfs))
market_spf = superfs[superfs['source'].isin(market_sources)]
print(len(market_spf))

957
79


In [9]:
spf_market_score = pd.merge(
    market_spf[['id', 'source', 'forecast']],
    market_res[['id', 'resolution_date', 'resolved_to', 'resolved']], 
    on=['id'],
    how='inner'
)
spf_market_score = spf_market_score[['id', 'source', 'forecast', 'resolution_date', 'resolved_to', 'resolved']]
print(len(spf_market_score))

54


In [20]:
spf_market_score['bs'] = (spf_market_score['forecast'] - spf_market_score['resolved_to']) **2
# superforecasters -- Brier score on market questions
print(spf_market_score['bs'].mean())
print(spf_market_score['bs'].median())

spf_market_score['resolved_to'].value_counts()

0.08191039004629627
0.000578125


0.0    40
1.0    14
Name: resolved_to, dtype: int64

### Question Source: Data

In [14]:
data_spf = superfs[superfs['source'].isin(data_sources)]
print(len(data_spf))

878


In [15]:
spf_data_score = pd.merge(
    data_spf[['id', 'resolution_date', 'source', 'forecast']],
    data_res[['id', 'resolution_date', 'resolved_to', 'resolved']], 
    on=['id', 'resolution_date'], # questions have the same id but different resolution dates
    how='inner'
)
spf_data_score = spf_data_score[['id', 'resolution_date', 'source', 'forecast', 'resolved_to', 'resolved']]
print(len(spf_data_score))

526


In [17]:
print("Questions with the same ID :")
print(spf_data_score['id'].duplicated().any())
print(spf_data_score['id'].value_counts().loc[lambda x: x > 1].count())

Questions with the same ID :
True
106


In [21]:
spf_data_score['bs'] = (spf_data_score['forecast'] - spf_data_score['resolved_to'])**2
# superforecasters -- Brier score on data questions
print(spf_data_score['bs'].mean())
print(spf_data_score['bs'].median())

spf_data_score['resolved_to'].value_counts()

0.12126039913498106
0.09


0.0    350
1.0    176
Name: resolved_to, dtype: int64

## General Public

### Question Source: Market

In [11]:
with open('../../ForecastBench-HumanGroups/2024-07-21.ForecastBench.human_public.json') as file:
    pub = json.load(file) #general public
gen_pub = pd.json_normalize(pub, record_path=['forecasts'])
print(len(gen_pub))
market_pub = gen_pub[gen_pub['source'].isin(market_sources)]
print(len(market_pub))

967
89


In [12]:
pub_market_score = pd.merge(
    market_pub[['id', 'source', 'forecast']],
    market_res[['id', 'resolution_date', 'resolved_to', 'resolved']], 
    on=['id'],
    how='inner'
)
pub_market_score = pub_market_score[['id', 'source', 'forecast', 'resolution_date', 'resolved_to', 'resolved']]
print(len(pub_market_score))

54


In [22]:
pub_market_score['bs'] = (pub_market_score['forecast'] - pub_market_score['resolved_to'])**2
# general public -- Brier score on market questions
print(pub_market_score['bs'].mean())
print(pub_market_score['bs'].median())

pub_market_score['resolved_to'].value_counts()

0.12238101851851851
0.025700000000000008


0.0    40
1.0    14
Name: resolved_to, dtype: int64

### Question Source: Data

In [23]:
data_pub = gen_pub[gen_pub['source'].isin(data_sources)]
print(len(data_pub))

878


In [24]:
pub_data_score = pd.merge(
    data_pub[['id', 'resolution_date', 'source', 'forecast']],
    data_res[['id', 'resolution_date', 'resolved_to', 'resolved']], 
    on=['id', 'resolution_date'],
    how='inner'
)
pub_data_score = pub_data_score[['id', 'resolution_date', 'source', 'forecast', 'resolved_to', 'resolved']]
print(len(pub_data_score))

526


In [25]:
pub_data_score['bs'] = (pub_data_score['forecast'] - pub_data_score['resolved_to'])**2
# general public -- Brier score on data questions
print(pub_data_score['bs'].mean())
print(pub_data_score['bs'].median())

pub_data_score['resolved_to'].value_counts()

0.15738521577946768
0.1242625


0.0    350
1.0    176
Name: resolved_to, dtype: int64

## Human Crowds Data

In [26]:
spf_market = spf_market_score.copy().rename(columns={'forecast': 'SPF_forecast', 'bs': 'SPF_bs'})
pub_market = pub_market_score.copy().rename(columns={'forecast': 'Pub_forecast', 'bs': 'Pub_bs'})
spf_data = spf_data_score.copy().rename(columns={'forecast': 'SPF_forecast', 'bs': 'SPF_bs'})
pub_data = pub_data_score.copy().rename(columns={'forecast': 'Pub_forecast', 'bs': 'Pub_bs'})

data_merged = spf_data.merge(
    pub_data[['id', 'resolution_date', 'Pub_forecast', 'Pub_bs']],
    on=['id', 'resolution_date'],
    how='inner'
)
data_merged['Type'] = 'Data'

market_merged = spf_market.merge(
    pub_market[['id', 'Pub_forecast', 'Pub_bs']],
    on='id',
    how='inner'
)
market_merged['Type'] = 'Market'

all_humans = pd.concat([data_merged, market_merged], ignore_index=True)
all_humans = all_humans[['id', 'resolution_date', 'source', 'Type', 'resolved_to', 'resolved', 'SPF_forecast', 'SPF_bs', 'Pub_forecast', 'Pub_bs']]

print(all_humans.shape)

(580, 10)


In [27]:
#all_humans.to_excel('extracted_fb_data/all_h_scores.xlsx', index=False)
#all_humans.to_csv('extracted_fb_data/all_h_scores.csv', index=False)

### Stats

In [28]:
cols = ['SPF_forecast', 'SPF_bs', 'Pub_forecast', 'Pub_bs']

# 1. Overall descriptive statistics
overall_stats = all_humans[cols].describe().round(3)
print("Overall Descriptive Statistics :")
print(overall_stats)

Overall Descriptive Statistics:
       SPF_forecast   SPF_bs  Pub_forecast   Pub_bs
count       580.000  580.000       580.000  580.000
mean          0.314    0.118         0.313    0.154
std           0.290    0.136         0.192    0.154
min           0.000    0.000         0.010    0.000
25%           0.007    0.000         0.130    0.016
50%           0.400    0.040         0.300    0.117
75%           0.515    0.240         0.500    0.250
max           1.000    0.951         0.950    0.902


In [32]:
# 2. Descriptive statistics by question source ('data', 'market')
type_stats = all_humans.groupby('Type')[cols].describe().round(3).T
print("Descriptive Statistics by questio  source :")
print(type_stats)


Descriptive Statistics by questio  source :
Type                   Data  Market
SPF_forecast count  526.000  54.000
             mean     0.316   0.300
             std      0.280   0.378
             min      0.000   0.000
             25%      0.006   0.010
             50%      0.415   0.050
             75%      0.511   0.579
             max      1.000   1.000
SPF_bs       count  526.000  54.000
             mean     0.121   0.082
             std      0.129   0.187
             min      0.000   0.000
             25%      0.000   0.000
             50%      0.090   0.001
             75%      0.240   0.023
             max      0.792   0.951
Pub_forecast count  526.000  54.000
             mean     0.310   0.347
             std      0.176   0.305
             min      0.010   0.010
             25%      0.150   0.062
             50%      0.300   0.260
             75%      0.500   0.550
             max      0.880   0.950
Pub_bs       count  526.000  54.000
             mean  

In [34]:
# 3. Descriptive statistics by resolved_to (0 or 1)
resolved_stats = all_humans.groupby('resolved_to')[cols].describe().round(3).T
print("Descriptive Statistics by resolution :")
print(resolved_stats)

Descriptive Statistics by resolution :
resolved_to             0.0      1.0
SPF_forecast count  390.000  190.000
             mean     0.175    0.599
             std      0.222    0.186
             min      0.000    0.110
             25%      0.000    0.500
             50%      0.020    0.530
             75%      0.444    0.665
             max      0.975    1.000
SPF_bs       count  390.000  190.000
             mean     0.080    0.195
             std      0.122    0.130
             min      0.000    0.000
             25%      0.000    0.112
             50%      0.000    0.221
             75%      0.197    0.250
             max      0.951    0.792
Pub_forecast count  390.000  190.000
             mean     0.237    0.471
             std      0.167    0.136
             min      0.010    0.078
             25%      0.100    0.400
             50%      0.200    0.500
             75%      0.374    0.500
             max      0.950    0.950
Pub_bs       count  390.000  190.000